<a href="https://colab.research.google.com/github/nishnarudkar/Hybrid-Financial-Market-Prediction-using-Big-Data-Analytics-Hidden-Markov-Models-and-LSTM-Networks/blob/main/notebooks/LSTM_model_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("hmm_regime_data.csv")

df.head()

In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
print(df.columns)

In [ ]:
# Convert Date
df["Date"] = pd.to_datetime(df["Date"])

# Sort time series
df = df.sort_values("Date")

df = df.reset_index(drop=True)

print("Dataset shape:", df.shape)

df.head()

In [ ]:
df["target_return"] = df["returns"].shift(-1)

df = df.dropna()

print("New shape:", df.shape)

In [ ]:
features = [
    "Close",
    "VIX",
    "returns",
    "volatility",
    "ma10",
    "ma50",
    "momentum",
    "RSI",
    "Regime"
]

target = "target_return"

In [ ]:
# Train-Test Split

train_size = int(0.8 * len(df))

train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

In [ ]:
# Normalize the Data

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(
    train_df[features]
)

test_scaled = scaler.transform(
    test_df[features]
)

scaled_data = np.vstack([
    train_scaled,
    test_scaled
])

In [ ]:
# Sequence Creation

sequence_length = 60

X = []
y = []

target_values = df[target].values

for i in range(sequence_length, len(scaled_data)):

    X.append(
        scaled_data[i-sequence_length:i]
    )

    y.append(
        target_values[i]
    )

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Split Sequences

split_index = int(0.8 * len(X))

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

In [ ]:
# Model Building

model = Sequential()

model.add(
    LSTM(
        units=64,
        return_sequences=True,
        input_shape=(
            X_train.shape[1],
            X_train.shape[2]
        )
    )
)

model.add(Dropout(0.2))

model.add(
    LSTM(
        units=64
    )
)

model.add(Dropout(0.2))

model.add(Dense(1))

model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

model.summary()

In [ ]:
# Train the Model

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend([
    "Train",
    "Validation"
])

plt.grid(True)

plt.show()

In [ ]:
predictions = model.predict(X_test)

print("Predictions shape:", predictions.shape)

In [ ]:
# Model Evaluation
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

mae = mean_absolute_error(
    y_test,
    predictions
)

r2 = r2_score(
    y_test,
    predictions
)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
# Model Plot Predictions

plt.figure(figsize=(10,5))

plt.plot(y_test)
plt.plot(predictions)

plt.title("Actual vs Predicted Returns")

plt.xlabel("Time")

plt.ylabel("Return")

plt.legend([
    "Actual",
    "Predicted"
])

plt.grid(True)

plt.show()

In [ ]:
# Direction Accuracy

pred = predictions.flatten()

y_true_direction = np.sign(y_test)
y_pred_direction = np.sign(pred)

directional_accuracy = np.mean(
    y_true_direction == y_pred_direction
)

print(
    "Directional Accuracy:",
    directional_accuracy
)